# 03 - Train a U-Net baseline
Splits **by geographic block across all dates** (no location appears in two splits). Use a GPU runtime.

**Important:** scores against weak labels only measure agreement with the thresholds. Report headline metrics on a test split whose labels you corrected by hand.

In [ ]:
import sys, warnings
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from hyacinthwatch.config import load_config
cfg = load_config("../config/config.yaml")
bbox = cfg["aoi"]["pilot_bbox"]
BANDS = cfg["bands"]
import torch
from hyacinthwatch.chips import spatial_split
from hyacinthwatch.model import channel_stats, train, evaluate, ChipDataset
from torch.utils.data import DataLoader

In [ ]:
t = cfg["training"]
index = pd.read_csv("../data/chips/index.csv")
index = spatial_split(index, block_m=t["block_m"])
print(index.split.value_counts())
mean, std = channel_stats(index[index.split == "train"].path)
np.savez("../models/stats.npz", mean=mean, std=std)

In [ ]:
model, hist = train(index, mean, std, in_channels=len(BANDS), encoder=t["encoder"], weights=t["weights"],
                    epochs=t["epochs"], batch_size=t["batch_size"], lr=t["lr"], neg_ratio=t["neg_ratio"],
                    out_path="../models/unet.pt")

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model.load_state_dict(torch.load("../models/unet.pt", map_location=device))
te = index[index.split == "test"]
print(evaluate(model.to(device), DataLoader(ChipDataset(te.path, mean, std), batch_size=16), device, t["threshold"]))
pd.DataFrame(hist).set_index("epoch")[["loss", "iou", "f1"]].plot(figsize=(8, 3.5)); plt.show()